### Dealing with Errors

Because SARFA relies on the use of the PyRadiomics library, which is a little outdated at this point (relies on Python 3.7 or older), we encountered some issues running the code in a Jupyter Notebook using modern conda versions. We did end up finding a solution to our issues, so if you run into any issues on your end trying to run this code, uncomment the cells below and run them through. This should fix any issue related to the error message 'libpython3.7m.so.1.0 not found' or similar messages.

In [ ]:
# import ctypes

# ctypes.CDLL("/path/to/your/libpython3.7m.so.1.0")
# print("loaded")

In [ ]:
# import subprocess

# subprocess.run(
#     [
#         "ldd",
#         "/path/to/your/python3.7/site-packages/radiomics/_cmatrices.cpython-37m-x86_64-linux-gnu.so"
#     ],
#     capture_output=True,
#     text=True
# ).stdout

In [ ]:
# pip show pyradiomics

### Imports

In [5]:
import os
import copy
import torch
import logging
import warnings
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import SimpleITK as sitk
from scipy.linalg import sqrtm
import torch.nn.functional as F
from torchvision import transforms
from typing import List, Tuple, Dict
from sam_lora_image_encoder import LoRA_Sam
from segment_anything import sam_model_registry
from torch.utils.data import DataLoader, Subset
from radiomics import featureextractor, setVerbosity
from load_LIDC_data import LIDC_IDRI, RandomGenerator
from utils import calculate_dice_loss, calculate_sigmoid_focal_loss

### Config

Please note that in order to run SARFA, you will need to have a SAM checkpoint saved on your device. In this work, we used the ViT-B checkpoint, which you can download from https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth. If you want to use any other checkpoint, such as ViT-H or ViT-L, you will likely have to modify this code to ensure architectural compatibility.

In [ ]:
CHECKPOINT_PATH   = "sam_vit_b_01ec64.pth"
BATCH_SIZE        = 1
LEARNING_RATE     = 1e-3
NUM_EPOCHS        = 100
IMG_SIZE          = 128

# DPO-specific
BETA              = 0.1        # DPO temperature
LAMBDA_DPO        = 0.05       # Weight of DPO loss relative to supervised losses
DPO_EVERY_K_STEPS = 10         # Run radiomics ranking every K steps

### Suppress Radiomics Logging

This step is optional. The code below (that is currently commented out) will reduce the amount of logging performed by PyRadiomics when you run the relevant pieces of code further in this file. There is A LOT of logging performed by PyRadiomics that makes it very hard to track things like training performance due to the sheer volume of messages sent to the console, so we highly recommend you uncomment the code below and run it.

In [ ]:
warnings.filterwarnings("ignore")
logging.getLogger("radiomics").setLevel(logging.CRITICAL)

for name in logging.Logger.manager.loggerDict:
    if "radiomics" in name.lower():
        logging.getLogger(name).setLevel(logging.CRITICAL)

setVerbosity(60)

### Mask Weights Module

In [ ]:
class MaskWeights(nn.Module):
    """Learnable weighted combination of the multiple SAM mask outputs."""
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.ones(7, 1, requires_grad=True) / 8)

### SAM Inference

In [ ]:
def inference(batched_input: torch.Tensor, lora_sam: LoRA_Sam) -> Dict[str, torch.Tensor]:
    input_images = lora_sam.sam.preprocess(batched_input)
    image_embeddings = lora_sam.sam.image_encoder(input_images)

    sparse_embeddings, dense_embeddings = lora_sam.sam.prompt_encoder(
        points=None, boxes=None, masks=None
    )

    low_res_masks, iou_predictions = lora_sam.sam.mask_decoder(
        image_embeddings=image_embeddings,
        image_pe=lora_sam.sam.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_embeddings,
        dense_prompt_embeddings=dense_embeddings,
        multimask_output=True
    )

    masks = lora_sam.sam.postprocess_masks(
        low_res_masks,
        input_size=(IMG_SIZE, IMG_SIZE),
        original_size=(IMG_SIZE, IMG_SIZE)
    )

    return {
        "masks": masks,                       # [B, 8, H, W]
        "iou_predictions": iou_predictions,   # [B, 8]
        "low_res_logits": low_res_masks       # [B, 8, H/4, W/4]
    }

def apply_mask_to_image(image: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Apply binary mask to image, keeping only masked region."""
    return image * mask.unsqueeze(0)

### Radiomics Integration

In [ ]:
def build_extractor() -> featureextractor.RadiomicsFeatureExtractor:
    extractor = featureextractor.RadiomicsFeatureExtractor()

    extractor.settings["binWidth"]          = 25
    extractor.settings["normalize"]         = False
    extractor.settings["removeOutliers"]    = 3
    extractor.settings["force2D"]           = True
    extractor.settings["force2Ddimension"]  = 0

    extractor.enableFeatureClassByName("firstorder")
    extractor.enableFeatureClassByName("glcm")
    extractor.enableFeatureClassByName("glrlm")
    extractor.enableFeatureClassByName("glszm")
    extractor.enableFeatureClassByName("gldm")
    extractor.enableFeatureClassByName("ngtdm")
    extractor.enableFeatureClassByName("shape2D")

    extractor.enableImageTypeByName("Original")
    extractor.enableImageTypeByName("Wavelet")

    return extractor

def extract_features_from_masked_image(
    image_tensor: torch.Tensor,
    mask_tensor: torch.Tensor,
    extractor
) -> Dict[str, float]:

    masked = apply_mask_to_image(image_tensor, mask_tensor)
    image_np = masked[0].cpu().numpy().astype(np.float32)
    mask_np  = mask_tensor.cpu().numpy().astype(np.uint8)

    image_sitk = sitk.GetImageFromArray(image_np)
    mask_sitk  = sitk.GetImageFromArray(mask_np)
    mask_sitk.CopyInformation(image_sitk)

    try:
        feats = extractor.execute(image_sitk, mask_sitk)
        return {
            k: float(v)
            for k, v in feats.items()
            if k.startswith("original") or k.startswith("wavelet")
        }
    except Exception:
        return {}

def features_to_vector(
    features: Dict[str, float],
    feature_names: List[str]
) -> np.ndarray:
    return np.array([features.get(n, 0.0) for n in feature_names], dtype=np.float32)

def compute_global_gt_stats(
    loader: DataLoader,
    extractor,
    device: torch.device
) -> Tuple[List[str], np.ndarray, np.ndarray]:
    """
    Pre-compute mean and std of radiomics features across all GT masks.
    Used for z-score normalisation when ranking per-sample masks.
    """
    images, masks = [], []

    for batch in tqdm(loader, desc="Computing GT radiomics stats"):
        img = batch["image"].to(device)
        lab = batch["label"].to(device)
        images.append(img[0])
        masks.append((lab[0] > 0).float())

    first_feats = extract_features_from_masked_image(images[0], masks[0], extractor)
    feature_names = sorted(first_feats.keys())

    X = np.zeros((len(images), len(feature_names)), dtype=np.float32)

    for i, (img, msk) in enumerate(zip(images, masks)):
        feats = extract_features_from_masked_image(img, msk, extractor)
        if feats:
            X[i] = features_to_vector(feats, feature_names)

    mu    = X.mean(axis=0)
    sigma = np.where(X.std(axis=0) < 1e-8, 1.0, X.std(axis=0))

    return feature_names, mu, sigma

def rank_masks_by_radiomics_distance(
    image: torch.Tensor,
    gt_mask: torch.Tensor,
    pred_masks: torch.Tensor,       # [8, H, W]
    extractor,
    feature_names: List[str],
    mu: np.ndarray,
    sigma: np.ndarray
) -> Tuple[int, int, List[float]]:
    """
    Returns the index of the chosen (best) and rejected (worst) mask
    based on z-scored MSE distance in radiomics feature space.
    """
    gt_feats = extract_features_from_masked_image(image, gt_mask, extractor)
    if not gt_feats:
        # GT features failed - return degenerate ranking
        return 0, 0, [float("inf")] * 8
    
    gt_vec   = features_to_vector(gt_feats, feature_names)
    gt_vec   = np.nan_to_num(gt_vec, nan=0.0, posinf=0.0, neginf=0.0)
    gt_z     = (gt_vec - mu) / sigma
    gt_z     = np.nan_to_num(gt_z, nan=0.0, posinf=0.0, neginf=0.0)

    distances: List[float] = []

    for i in range(8):
        feats = extract_features_from_masked_image(image, pred_masks[i], extractor)
        if not feats:
            distances.append(float("inf"))
            continue
        vec = features_to_vector(feats, feature_names)
        vec = np.nan_to_num(vec, nan=0.0, posinf=0.0, neginf=0.0)
        z   = (vec - mu) / sigma
        z   = np.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
        
        dist = np.mean((z - gt_z) ** 2)
        dist = np.nan_to_num(dist, nan=float("inf"), posinf=float("inf"), neginf=float("inf"))
        distances.append(float(dist))

    # If all distances are inf or identical, return degenerate ranking
    finite_dists = [d for d in distances if np.isfinite(d)]
    if len(finite_dists) < 2 or len(set(finite_dists)) == 1:
        return 0, 0, distances

    return int(np.argmin(distances)), int(np.argmax(distances)), distances

### DPO Loss

In [ ]:
def dpo_loss(
    iou_pi: torch.Tensor,    # [B, 8] — policy model IoU predictions
    iou_ref: torch.Tensor,   # [B, 8] — frozen reference model IoU predictions
    chosen: torch.Tensor,    # [B]    — index of radiomically preferred mask
    rejected: torch.Tensor,  # [B]    — index of radiomically worst mask
    beta: float
) -> torch.Tensor:
    """
    DPO loss over IoU prediction scores.
    Trains the policy to score the radiomically-preferred mask
    higher than the rejected one, relative to the frozen reference.
    """
    logp_pi  = torch.log_softmax(iou_pi,  dim=1)
    logp_ref = torch.log_softmax(iou_ref, dim=1)

    pi_diff = (
        logp_pi.gather(1, chosen.unsqueeze(1)).squeeze(1)
        - logp_pi.gather(1, rejected.unsqueeze(1)).squeeze(1)
    )

    ref_diff = (
        logp_ref.gather(1, chosen.unsqueeze(1)).squeeze(1)
        - logp_ref.gather(1, rejected.unsqueeze(1)).squeeze(1)
    )

    return -F.logsigmoid(beta * (pi_diff - ref_diff)).mean()

### FRD Utilities

In [ ]:
def extract_features_from_dataset(images, masks, extractor, desc="Extracting features"):
    first_features = extract_features_from_masked_image(images[0], masks[0], extractor)
    if not first_features:
        return None, None

    feature_names = sorted(first_features.keys())
    X = np.zeros((len(images), len(feature_names)), dtype=np.float32)

    for i, (image, mask) in enumerate(tqdm(zip(images, masks), total=len(images), desc=desc)):
        feats = extract_features_from_masked_image(image, mask, extractor)
        if feats:
            X[i] = np.array([feats.get(n, 0.0) for n in feature_names])

    return X, feature_names


def zscore_normalize_reference(X_ref, X_cmp, eps=1e-8):
    X_ref = np.nan_to_num(X_ref)
    X_cmp = np.nan_to_num(X_cmp)
    mu    = X_ref.mean(axis=0)
    sigma = X_ref.std(axis=0)
    sigma = np.where(sigma < eps, 1.0, sigma)
    X_ref_z = np.nan_to_num((X_ref - mu) / sigma)
    X_cmp_z = np.nan_to_num((X_cmp - mu) / sigma)
    return X_ref_z, X_cmp_z, mu, sigma


def frechet_distance(mu1, cov1, mu2, cov2, eps=1e-6):
    mu1, mu2 = np.float64(np.nan_to_num(mu1)), np.float64(np.nan_to_num(mu2))
    cov1 = np.float64(np.nan_to_num(cov1))
    cov2 = np.float64(np.nan_to_num(cov2))
    diff = mu1 - mu2
    m    = cov1.shape[0]
    try:
        covmean = sqrtm((cov1 + eps * np.eye(m)) @ (cov2 + eps * np.eye(m)))
        if np.iscomplexobj(covmean):
            covmean = covmean.real
        covmean = np.nan_to_num(covmean)
        d = float(max((diff @ diff) + np.trace(cov1) + np.trace(cov2) - 2 * np.trace(covmean), 0.0))
    except Exception:
        d = float("inf")
    return d


def frd_from_zscored(X_ref_z, X_cmp_z, eps=1e-6):
    mu_ref,  cov_ref  = X_ref_z.mean(axis=0), np.cov(X_ref_z, rowvar=False, bias=False)
    mu_cmp,  cov_cmp  = X_cmp_z.mean(axis=0), np.cov(X_cmp_z, rowvar=False, bias=False)
    cov_ref = np.nan_to_num(cov_ref)
    cov_cmp = np.nan_to_num(cov_cmp)
    d_f = frechet_distance(mu_ref, cov_ref, mu_cmp, cov_cmp, eps=eps)
    frd = float(np.log(d_f)) if d_f > 0.0 else float("-inf")
    return frd, d_f


def calculate_frd_for_epoch(all_images, all_gt_masks, all_pred_masks, extractor):
    X_gt, feature_names = extract_features_from_dataset(
        all_images, all_gt_masks, extractor, desc="GT features (FRD)"
    )
    if X_gt is None:
        return None

    frd_scores = []
    for mask_idx in range(8):
        masks_at_idx = [pm[mask_idx] for pm in all_pred_masks]
        X_pred, _ = extract_features_from_dataset(
            all_images, masks_at_idx, extractor, desc=f"Mask {mask_idx} features (FRD)"
        )
        if X_pred is None:
            frd_scores.append(float("inf"))
            continue
        X_gt_z, X_pred_z, _, _ = zscore_normalize_reference(X_gt, X_pred)
        frd_value, _ = frd_from_zscored(X_gt_z, X_pred_z)
        frd_scores.append(frd_value)

    return frd_scores


def print_frd_summary(frd_scores, epoch, avg_loss, dpo_steps):
    best_mask_idx = int(np.argmin(frd_scores))
    print(f"\n{'='*70}")
    print(
        f"EPOCH {epoch+1} COMPLETE | "
        f"Avg Loss: {avg_loss:.4f} | "
        f"DPO steps: {dpo_steps} | "
        f"Best Mask: {best_mask_idx} (FRD: {frd_scores[best_mask_idx]:.4f})"
    )
    print(f"{'='*70}")
    print(f"{'Mask':<8}" + "".join([f"{i:<10}" for i in range(8)]))
    print(f"{'FRD':<8}" + "".join([f"{s:<10.4f}" for s in frd_scores]))
    print(f"{'='*70}\n")

### Training

In [ ]:
def train_hybrid(
    checkpoint=CHECKPOINT_PATH,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    num_epochs=NUM_EPOCHS,
    beta=BETA,
    lambda_dpo=LAMBDA_DPO,
    dpo_every_k_steps=DPO_EVERY_K_STEPS,
):
    device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    print(f"DPO config — beta: {beta}, lambda: {lambda_dpo}, every K={dpo_every_k_steps} steps")

    # ------------------------------------------------------------------
    # Build model
    # ------------------------------------------------------------------
    sam, img_embedding_size = sam_model_registry["vit_b"](
        image_size=IMG_SIZE,
        num_classes=8,
        pixel_mean=[0, 0, 0],
        pixel_std=[1, 1, 1],
        checkpoint=checkpoint
    )

    low_res = img_embedding_size * 4

    # Policy model — trained
    policy = LoRA_Sam(sam, 4).to(device)

    for p in policy.sam.prompt_encoder.parameters():
        p.requires_grad = True
    for p in policy.sam.mask_decoder.parameters():
        p.requires_grad = True

    # Reference model — frozen copy of policy at init
    ref = copy.deepcopy(policy).to(device)
    for p in ref.parameters():
        p.requires_grad = False
    ref.eval()

    # Learnable mask weighting head
    mask_weights = MaskWeights().to(device)

    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    dataset = LIDC_IDRI(
        dataset_location="data/",
        transform=transforms.Compose([
            RandomGenerator(
                output_size=[IMG_SIZE, IMG_SIZE],
                low_res=[low_res, low_res],
                test=True
            )
        ])
    )

    dataset_size  = len(dataset)
    train_split   = int(0.6 * dataset_size)
    train_loader  = DataLoader(
        Subset(dataset, list(range(train_split))),
        batch_size=batch_size,
        shuffle=True
    )

    print(f"Dataset size: {dataset_size} | Training set: {train_split}")

    # ------------------------------------------------------------------
    # Optimizers
    # ------------------------------------------------------------------
    optimizer1 = torch.optim.Adam(
        filter(lambda p: p.requires_grad, policy.parameters()),
        lr=lr, weight_decay=0
    )
    optimizer2 = torch.optim.AdamW(
        mask_weights.parameters(),
        lr=lr, eps=1e-4
    )

    # ------------------------------------------------------------------
    # Radiomics setup
    # ------------------------------------------------------------------
    extractor = build_extractor()

    # Pre-compute GT feature statistics for per-step DPO ranking
    feature_names, mu, sigma = compute_global_gt_stats(train_loader, extractor, device)

    # ------------------------------------------------------------------
    # Training loop
    # ------------------------------------------------------------------
    os.makedirs("checkpoints", exist_ok=True)

    all_frd_scores = []
    best_frd       = float("inf")
    best_epoch     = -1
    global_step    = 0

    for epoch in range(num_epochs):
        policy.train()
        mask_weights.train()

        epoch_loss      = 0.0
        epoch_sup_loss  = 0.0
        epoch_dpo_loss  = 0.0
        num_batches     = 0
        dpo_steps       = 0

        all_images    = []
        all_gt_masks  = []
        all_pred_masks = []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")

        for batch_idx, sampled_batch in enumerate(pbar):
            image_batch = sampled_batch["image"].to(device)
            label_batch = sampled_batch["label"].to(device)

            # ---- Forward pass (policy) --------------------------------
            out_pi = inference(image_batch, policy)
            logits_high = out_pi["masks"]   # [B, 8, H, W]

            # ---- Weighted combination for supervised losses -----------
            weights_eight = torch.cat(
                (1 - mask_weights.weights.sum(0).unsqueeze(0),
                 mask_weights.weights),
                dim=0
            ).to(device)

            logits_high_weighted = logits_high * weights_eight.unsqueeze(-1)
            logits_high_res      = logits_high_weighted.sum(1).unsqueeze(1)

            # ---- Supervised losses ------------------------------------
            ce_loss    = nn.CrossEntropyLoss()(logits_high, label_batch.long())
            gt_mask_4d = label_batch.unsqueeze(1)
            dice_loss  = calculate_dice_loss(logits_high_res, gt_mask_4d.long())
            focal_loss = calculate_sigmoid_focal_loss(logits_high_res, gt_mask_4d.float())

            sup_loss = ce_loss + dice_loss + focal_loss

            # ---- DPO loss (every K steps) -----------------------------
            loss_dpo = torch.tensor(0.0, device=device)

            if global_step % dpo_every_k_steps == 0:
                # Binarise predicted masks for radiomics
                pred_masks_bin = (logits_high[0] > 0).float().detach()
                gt_bin         = (label_batch[0] > 0).float()

                chosen_idx, rejected_idx, distances = rank_masks_by_radiomics_distance(
                    image_batch[0], gt_bin, pred_masks_bin,
                    extractor, feature_names, mu, sigma
                )
                
                # Only compute DPO if we got a valid ranking
                if chosen_idx != rejected_idx:
                    # Reference forward pass (no grad)
                    with torch.no_grad():
                        out_ref = inference(image_batch, ref)

                    loss_dpo = dpo_loss(
                        out_pi["iou_predictions"],
                        out_ref["iou_predictions"],
                        torch.tensor([chosen_idx],  device=device),
                        torch.tensor([rejected_idx], device=device),
                        beta=beta
                    )
                    dpo_steps += 1

            # ---- Combined loss ----------------------------------------
            total_loss = sup_loss + lambda_dpo * loss_dpo

            # ---- Backward pass ----------------------------------------
            optimizer1.zero_grad()
            optimizer2.zero_grad()
            total_loss.backward()
            optimizer1.step()
            optimizer2.step()

            # ---- Bookkeeping -----------------------------------------
            epoch_loss     += total_loss.item()
            epoch_sup_loss += sup_loss.item()
            epoch_dpo_loss += loss_dpo.item()
            num_batches    += 1
            global_step    += 1

            all_images.append(image_batch[0].detach())
            all_gt_masks.append((label_batch[0] > 0).float().detach())
            all_pred_masks.append((logits_high[0] > 0).float().detach())

            pbar.set_postfix(
                total=f"{total_loss.item():.4f}",
                sup=f"{sup_loss.item():.4f}",
                dpo=f"{loss_dpo.item():.4f}"
            )

        avg_loss = epoch_loss / num_batches

        # ---- Epoch-level FRD evaluation ------------------------------
        frd_scores = calculate_frd_for_epoch(
            all_images, all_gt_masks, all_pred_masks, extractor
        )

        if frd_scores is not None:
            all_frd_scores.append(frd_scores)
            print_frd_summary(frd_scores, epoch, avg_loss, dpo_steps)


            best_mask_idx    = int(np.argmin(frd_scores))
            current_best_frd = frd_scores[best_mask_idx]

            if current_best_frd < best_frd:
                best_frd   = current_best_frd
                best_epoch = epoch + 1

                torch.save(
                    {
                        "epoch":                  epoch + 1,
                        "lora_sam_state_dict":    policy.state_dict(),
                        "mask_weights_state_dict": mask_weights.state_dict(),
                        "optimizer1_state_dict":  optimizer1.state_dict(),
                        "optimizer2_state_dict":  optimizer2.state_dict(),
                        "best_frd":               best_frd,
                        "best_mask_idx":          best_mask_idx,
                        "frd_scores":             frd_scores,
                        "avg_loss":               avg_loss,
                        # Save DPO config so evaluation script can inspect it
                        "dpo_config": {
                            "beta":              beta,
                            "lambda_dpo":        lambda_dpo,
                            "dpo_every_k_steps": dpo_every_k_steps,
                        },
                    },
                    "checkpoints/frd_convergence.pth"
                )
                print(f"*** New best model saved! FRD: {best_frd:.4f} (Mask {best_mask_idx}) ***\n")

    # ---- Final summary -----------------------------------------------
    print("\n" + "=" * 70)
    print("TRAINING COMPLETE — FRD PROGRESSION ACROSS EPOCHS")
    print("=" * 70)
    print(f"\n{'Epoch':<10}" + "".join([f"{'Mask ' + str(i):<12}" for i in range(8)]) + "Best Mask")
    print("-" * 120)

    for ep, scores in enumerate(all_frd_scores):
        best_idx = int(np.argmin(scores))
        print(f"{ep+1:<10}" + "".join([f"{s:<12.4f}" for s in scores]) + str(best_idx))

    print("=" * 70)
    print(f"\nBest model saved at epoch {best_epoch} with FRD: {best_frd:.4f}")
    print("Checkpoint: checkpoints/frd_convergence.pth")
    print("=" * 70)

    return all_frd_scores


if __name__ == "__main__":
    all_frd_scores = train_hybrid()